## Lokales RAG mit LMStudio

Immer mal wieder gibt es Fälle, bei denen ihr die Daten nicht einfach an ein Cloud-LLM schicken könnt. Z.B. bei personenbezogenen Daten, geschützten Firmendaten oder auch Benotung und Gutachten für Bachelorarbeiten ;-). In solchen Fällen bietet sich RAG mit lokal laufendem VectorStore und LLM an. Das möchten wir heute sehr rudimentär mit LMStudio (Modelhosting), Chroma (VectorStore) und dem Llama 3.2-3B Instruct Modell umsetzen, um ein PDF zu parsen und als Kontext mit dem Query mit zu schicken.

Wir installieren zuerst die notwendigen Pakete:

In [ ]:
pip install langchain_community langchain[openai] langchain-chroma pypdf chromadb

Wir setzen einige Konstanten.

Als PDF am besten ein Buch, Bachelorarbeit oder ähnliches nutzen. Es sollte mindestens 50 Seiten Inhalt haben.

In [ ]:
URL_LLM_SERVER = "http://localhost:1234"
MODEL_NAME = "llama-3.2-3b-instruct"
EMBDEDDING_MODEL_NAME = "text-embedding-nomic-embed-text-v1.5"

CHROMA_DB_DIR = "./chroma_db"
PDF_PATH = "TODO.pdf"

### LMStudio installieren und Modelle herunterladen

Für das Hosting unserer Modelle nutzen wir https://lmstudio.ai/. Es bietet uns OpenAI kompatible REST Endpoints, so dass wir lokale Modelle mit derselben Implementierung wie Cloud-Modelle nutzen können. Ladet euch für die Übung vorerst folgende Modelle herunter:
- Llama-3.2
- text-embedding-nomic-embed

Anschließend können wir im DeveloperTab (!in der unteren Zeile muss "Power User" ausgewählt sein!) den Modell-Server starten und die beiden Modelle laden. Sofern alles richtig aufgesetzt ist müssten folgende REST-Anfragen an den LMStudio Server funktionieren:

In [ ]:
# Request to embedding model

import requests
import json

headers = {
    "Content-Type": "application/json",
}
payload = {
    "model": EMBDEDDING_MODEL_NAME,
    "input": "Ich möchte diesen Text als Embedding Vektor haben"
}

embedding_response = requests.post(URL_LLM_SERVER + "/v1/embeddings", headers=headers, data=json.dumps(payload))
if embedding_response.status_code == 200:
    result = embedding_response.json()
    print(json.dumps(result, indent=2))
else:
    print(f"Fehler bei der Anfrage: {embedding_response.status_code}")
    print(embedding_response.text)

In [101]:
# Request to llama model

import requests
import json

query = "Was ist RAG?"
headers = {
    "Content-Type": "application/json",
}
payload = {
    "model": "llama-3.2-3b-instruct",
    "messages": [
        {"role": "system", "content": "Du bist ein hilfreicher Assistent."},
        {"role": "user", "content": query}
    ],
    "temperature": 0.7,
    "max_tokens": -1,
    "stream": False
}

response = requests.post(URL_LLM_SERVER + "/v1/chat/completions", headers=headers, data=json.dumps(payload))
if response.status_code == 200:
    result = response.json()
    print(json.dumps(result, indent=2))
else:
    print(f"Fehler bei der Anfrage: {response.status_code}")
    print(response.text)


{
  "id": "chatcmpl-cn1x2i0ma4bnb17nsk8s0b",
  "object": "chat.completion",
  "created": 1766064551,
  "model": "llama-3.2-3b-instruct",
  "choices": [
    {
      "index": 0,
      "logprobs": null,
      "finish_reason": "stop",
      "message": {
        "role": "assistant",
        "content": "\"RAG\" kann mehrere Dinge bedeuten, abh\u00e4ngig vom Kontext. Hier sind einige m\u00f6gliche Bedeutungen:\n\n1. **Reichsautomobil-Gesellschaft (RAG)**: RAG ist ein deutsches Automobilhersteller, der speziell auf die Produktion von Lkw und Busse spezialisiert ist. Die Firma wurde 1922 gegr\u00fcndet und hat ihren Sitz in Stuttgart, Deutschland.\n2. **Ressourcenzentrum f\u00fcr Gesundheit (RAG)**: RAG ist ein deutsches Unternehmen, das sich auf die Entwicklung und den Vertrieb von Gesundheits- und Wellness-L\u00f6sungen spezialisiert hat.\n3. **Rheinisch-Aachener Gasverkauf (RAG)**: RAG ist ein ehemaliger deutscher Energiekonzern, der 1963 gegr\u00fcndet wurde. Das Unternehmen hat sich im Lau

### PDF Laden und im Vektorstore speichern

Mit der Bib PyPDF können wir den Inhalt des PDFs laden:

In [102]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

Wir unterteilen das PDF in kleinere Chunks. Hier könnt ihr mit der chunk_size und chunk_overlap etwas spielen, initial würde ich mit ```chunk_size=500, chunk_overlap=50``` starten.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(pages)
chunks[0]

Um nun die Chunks mit ihren entsprechenden Embeddings in den VectoreStore (ChromaDB) zu bekommen, gibt uns langchain eine Funktion ```Chroma.from_documents(chunks, embeddings, ..)```. Chunks sind dabei unsere Texte aus dem PDF-Dokument. embeddings ist etwas komplexer, hier müssen wir eine Instanz einer Klasse angeben, welche zwei Funktionen implementiert: ```embed_documents``` und ```embed_query```. Um die Embeddings mehrerer Chunks zu generieren, ruft die ```Chroma.from_documents``` Funktion ```embed_docuemts``` auf und für die Generierung von Embeddings für ein Query ```embed_query```. Für die gängigen Cloud-LLMs bietet LangChain entsprechende Implementierungen, bei welchen man nur einen API_KEY angeben muss. Da wir jedoch ein lokales Embedding Modell nutzen wollen, müssen wir uns diese Klasse mit entsprechendem REST Request selbst schreiben:

In [ ]:
import requests
from typing import List

class LMStudioEmbeddings:
    def __init__(self, model):
        self.model = model

    def _req_lm_studio(self, chunk: str) -> List[float]:
        headers = {
            "Content-Type": "application/json",
        }
        payload = {
            "model": EMBDEDDING_MODEL_NAME,
            "input": chunk
        }
        embedding_res = requests.post(URL_LLM_SERVER + "/v1/embeddings", headers=headers, data=json.dumps(payload))
        return embedding_res.json()['data'][0]['embedding']

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self._req_lm_studio(t) for t in texts]
            
    def embed_query(self, query: str) -> List[float]:
        return self._req_lm_studio(query)
    
embeddings = LMStudioEmbeddings(model=EMBDEDDING_MODEL_NAME)

In [105]:
from langchain_chroma import Chroma

db_chroma = Chroma.from_documents(chunks, embeddings, persist_directory=CHROMA_DB_DIR)

### Retrieve und LLM Query

Wir holen uns aus dem VektorStore die ähnlichsten Chunks zu unserer Anfrage und bauen uns daraus einen Kontext für die Anfrage an das LLM.

Spielt hier mit der Query und dem Parameter ```k=20```, welcher die Anzahl der ähnlichsten Dokumente zur Query bestimmt. Ein zu kleines k bewirkt, dass wir nur sehr wenig Kontext dem LLM geben, ein zu großes k kostet uns Tokens bzw. wir landen über der Kontextlänge eines Modells.

In [106]:
query = 'Was ist der Forschungsstand?'

docs_chroma = db_chroma.similarity_search_with_score(query, k=20)

context_text = "\n\n".join([doc.page_content for doc, _score in docs_chroma])

In [ ]:
print(context_text)

Und nun schicken wir das Query mit dem entsprechenden Kontext an das lokale LLM :)

In [108]:
import requests
import json

system_prompt = f"Du bist ein hilfreicher Assistent. Beantworte die Fragen basierend auf folgendem Kontext: {context_text}. Gebe eine detaillierte Antwort. Rechtfertige deine Antwort nicht. Gebe keine Information die nicht im Kontext enthalten ist. Sage nicht 'laut dem Kontext' oder 'im Kontext erwähnt' oder ähnliches. Antworte in Deutsch"

headers = {
    "Content-Type": "application/json",
}

payload = {
    "model": "llama-3.2-3b-instruct",
    "messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ],
    "temperature": 0.7,
    "max_tokens": -1,
    "stream": False
}

response = requests.post(URL_LLM_SERVER + "/v1/chat/completions", headers=headers, data=json.dumps(payload))

In [ ]:
if response.status_code == 200:
    result = response.json()
    print(json.dumps(result, indent=2))  # Formatierte Ausgabe der Antwort
else:
    print(f"Fehler bei der Anfrage: {response.status_code}")
    print(response.text)

In der heutigen Aufgabe gab es wenig zu programmieren. Wichtig sind:
1) Den gegebenen Code verstehen
2) Verschiedene PDF-Dokumente ausprobieren
3) Parameter verändern: ChunkSize, k
4) Verschiedene Querys abschicken

Optional kann man gerne versuchen ein anderes LLM in LMStudio zu nutzen.